# 🚀 [ICML 2026] LiDAR: Lookahead Sample Reward Guidance for Diffusion Models
### Reproduce Table 2: `LiDAR (DPM-5 / n=50)` on GenEval Benchmark with Google Drive Persistence

Paper: *Lookahead Sample Reward Guidance for Test-Time Scaling of Diffusion Models* ([arXiv:2602.03211](https://arxiv.org/abs/2602.03211))  
Official Repository: [github.com/aailab-kaist/Diffusion-LiDAR-Sampling](https://github.com/aailab-kaist/Diffusion-LiDAR-Sampling)  
Target Benchmark: **GenEval (553 prompts)** with **ImageReward (IR), CLIP-Score, and HPS v2.1**.

## 📌 Step 1: Check GPU & Mount Google Drive
Kết nối Google Drive để toàn bộ kết quả ảnh và điểm số định lượng được lưu vĩnh viễn (không bị mất khi Colab đóng runtime).

In [ ]:
# 1. Kiểm tra GPU
!nvidia-smi

# 2. Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')

# 3. Thư mục lưu kết quả trên Google Drive
GDRIVE_DIR = "/content/drive/MyDrive/LiDAR_Experiment"
os.makedirs(GDRIVE_DIR, exist_ok=True)
print(f"\n✅ Đã kết nối Google Drive! Dữ liệu sẽ lưu tại: {GDRIVE_DIR}")

## 📦 Step 2: Clone Repository & Install Dependencies

In [ ]:
# 1. Quay về thư mục gốc /content trước khi clone
%cd /content

# 2. Cập nhật hoặc clone mã nguồn mới nhất
import os
if os.path.exists("/content/Noisy-Reward"):
    %cd /content/Noisy-Reward
    !git pull origin main
else:
    !git clone https://github.com/leekwanreal/Noisy-Reward.git /content/Noisy-Reward
    %cd /content/Noisy-Reward

# 3. Cài đặt các thư viện cần thiết (chuẩn transformers==4.38.2 cho ImageReward)
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.38.2 diffusers==0.31.0 accelerate safetensors huggingface-hub einops ftfy timm
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q git+https://github.com/THUDM/ImageReward.git
!pip install -q hpsv2 matplotlib tqdm scipy seaborn pandas

# 4. Tải bổ sung file vocab cho hpsv2 (khắc phục lỗi thiếu file của PyPI hpsv2)
import urllib.request, hpsv2
hpsv2_vocab = os.path.join(os.path.dirname(hpsv2.__file__), "src", "open_clip", "bpe_simple_vocab_16e6.txt.gz")
os.makedirs(os.path.dirname(hpsv2_vocab), exist_ok=True)
if not os.path.exists(hpsv2_vocab):
    urllib.request.urlretrieve("https://github.com/openai/CLIP/raw/main/clip/bpe_simple_vocab_16e6.txt.gz", hpsv2_vocab)
print("\n✅ Môi trường và các Reward Model đã sẵn sàng 100%!")

## ⚡ Step 3: Phase 1 — Lookahead Sampling (DPM-5, n=50 particles)
- Sử dụng **Stable Diffusion v1.5** với bộ giải **DPMSolver (5 steps)**.
- Sinh **50 particles** mẫu cho mỗi prompt và đánh giá trước hàm thưởng **ImageReward**.
- *(Thời gian chạy: ~15 phút trên Colab T4 cho toàn bộ 553 prompt)*.

In [ ]:
# Chạy Lookahead Sampling (Pha 1)
!python lookahead_sampling.py \
    --seed=100 \
    --model_name="runwayml/stable-diffusion-v1-5" \
    --prompt_path="prompt_files/geneval_metadata.jsonl" \
    --output_dir="/content/drive/MyDrive/LiDAR_Experiment/Lookahead_samples" \
    --num_particles=50 \
    --num_inference_steps=5 \
    --metrics_to_compute="ImageReward#Clip-Score#HumanPreference" \
    --save_individual_images=True

## 🎯 Step 4: Phase 2 — LiDAR Target Sampling (DDIM 50-steps / DDPM 100-steps)
- Lấy mẫu hướng theo phân phối Target với Reward Guidance scale $w=12.5$.
- Sinh song song **4 ảnh / prompt** (chuẩn đánh giá GenEval trong Bảng 2 của bài báo).
- *(Thời gian chạy: ~25 phút cho bản 50-step hoặc ~40 phút cho bản 100-step)*.

In [ ]:
# Chạy LiDAR Sampling (Pha 2) - DDIM 50 steps
!python LiDAR_sampling.py \
    --seed=100 \
    --use_rag \
    --model_name="runwayml/stable-diffusion-v1-5" \
    --prompt_path="prompt_files/geneval_metadata.jsonl" \
    --output_dir="/content/drive/MyDrive/LiDAR_Experiment/Target_samples" \
    --num_inference_steps=50 \
    --num_particles=4 \
    --top_k=50 \
    --scale=12.5 \
    --resample_t_end=200 \
    --lookahead_path="/content/drive/MyDrive/LiDAR_Experiment/Lookahead_samples/100_50_5" \
    --metrics_to_compute="ImageReward#Clip-Score#HumanPreference" \
    --save_individual_images

## 📊 Step 5: Đọc kết quả & So sánh với Bảng 2 trong bài báo

In [ ]:
import json
import glob
import os
from PIL import Image
import matplotlib.pyplot as plt

# 1. Đọc số liệu kết quả từ Google Drive
target_runs = sorted(glob.glob("/content/drive/MyDrive/LiDAR_Experiment/Target_samples/*"))
if not target_runs:
    raise FileNotFoundError("Chưa tìm thấy thư mục kết quả. Hãy đảm bảo Step 4 đã chạy xong!")

latest_dir = target_runs[-1]
metrics_path = os.path.join(latest_dir, "final_metrics.json")

with open(metrics_path, "r") as f:
    metrics = json.load(f)

print("="*65)
print("📈 KẾT QUẢ THỰC NGHIỆM VS BÀI BÁO (TABLE 2 - SD v1.5 LiDAR DPM-5/n=50)")
print("="*65)
print(f"• ImageReward (IR):        {metrics['ImageReward']['mean']:.4f}  | Bài báo: 0.378 ~ 0.384")
print(f"• CLIP Score:             {metrics['Clip-Score']['mean']:.4f}  | Bài báo: 0.278")
print(f"• HumanPreference (HPS):  {metrics['HumanPreference']['mean']:.4f}  | Bài báo: 0.276 ~ 0.277")
print("="*65)

# 2. Hiển thị ảnh mẫu sinh ra
sample_grid = os.path.join(latest_dir, "00000/grid.png")
if os.path.exists(sample_grid):
    plt.figure(figsize=(16, 5))
    plt.imshow(Image.open(sample_grid))
    plt.axis("off")
    plt.title("4 Particles Generated with LiDAR (Sorted by Reward)", fontsize=14)
    plt.show()
else:
    print("Ảnh grid prompt 00000 không tồn tại.")